In [0]:
import time

import pyspark.sql.functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

CATALOG = 'car_workshop'
LAB = f'{CATALOG}.lab'

tables = spark.sql(f'SHOW TABLES IN {LAB}').collect()

for table_row in tables:
    spark.sql(f'DROP TABLE IF EXISTS {LAB}.{table_row.tableName}')
    print(f"dropped {table_row}")


spark.sql(f'DROP VOLUME IF EXISTS {LAB}.files')
spark.sql(f'DROP SCHEMA IF EXISTS {LAB}')
print(f"{LAB} is now clear, ready for recreation")

spark.sql(f'CREATE SCHEMA IF NOT EXISTS {LAB}')
spark.sql(f'CREATE VOLUME IF NOT EXISTS {LAB}.files')
LAB_DIR = f'/Volumes/{CATALOG}/lab/files'

print(f"{LAB} is created")


def timed(label, fn):
    t0 = time.time()
    result = fn()
    print(f'{label}: {time.time() - t0:.1f}s')
    return result


print(f'lab schema: {LAB}, lab volume: {LAB_DIR}')

dbutils.widgets.dropdown("Is cluster mode?", "False", ["True","False"])
is_cluster_mode = dbutils.widgets.get("Is cluster mode?")

In [0]:
# build an artificially skewed table for the exercises: 80% of rows -> one hot product
items = spark.table(f'{CATALOG}.fact.fact_sales_items')
hot_product = items.groupBy('product_id').count().orderBy(F.desc('count')).first()['product_id']

(items
 .withColumn('product_id',
             F.when(F.rand(seed=42) < 0.8, F.lit(hot_product)).otherwise(F.col('product_id')))
 .write.mode('overwrite').saveAsTable(f'{LAB}.skewed_sales_items'))

skewed = spark.table(f'{LAB}.skewed_sales_items')
display(skewed.groupBy('product_id').agg(F.count('*').alias('cnt')).orderBy(F.desc('cnt')).limit(5))

In [0]:
fact_schema = f"{CATALOG}.fact"
tables = spark.sql(f'SHOW TABLES IN {fact_schema}').collect()

for table_row in tables:
    table = spark.sql(f'DESCRIBE TABLE {fact_schema}.{table_row.tableName}')
    print(f"table: {table_row}")
    display(table)

In [0]:
%sql
select distinct sale_date from car_workshop.fact.fact_invoices 
order by sale_date asc

In [0]:
%sql
create table car_workshop.lab.fact_invoices as
with cte as (
  select 
    *,
    month(sale_date) as sales_month,
    case when month(sale_date) between 1 and 8  then 'skew_1'
    when month(sale_date) between 8 and 9 then 'skew_2'
    when month(sale_date) between 10 and 11 then 'skew_3'
    when month(sale_date) between 12 and 12 then 'skew_4'
    end as skew_test
  from car_workshop.fact.fact_invoices
),
skew_1_multiplied as (
  select cte.* 
  from cte
  CROSS JOIN LATERAL explode(sequence(1, 150)) AS t(multiplier)
  where skew_test = 'skew_1'
),
other_skews as (
  select *
  from cte
  where skew_test != 'skew_1'
),
combined as (
  select * from skew_1_multiplied
  union all
  select * from other_skews
)
-- check the skew
-- select skew_test, count(skew_test) as row_count 
-- from combined
-- group by skew_test
select * from combined

In [0]:
%sql
select skew_test, count(*) from car_workshop.lab.fact_invoices
group by skew_test

In [0]:
%sql
select count(*) from car_workshop.lab.fact_invoices

In [0]:
from pyspark.sql.window import Window

inv = spark.table(f'{LAB}.fact_invoices')

# pathological: ALL 643243725 rows of skew_1 must land in ONE task and get sorted there
w_skew = Window.partitionBy('skew_test').orderBy('sale_date')
timed('window over skew_test (4 tasks, one with 643243725 rows)',
      lambda: inv.withColumn('rn', F.row_number().over(w_skew))
                 .agg(F.max('rn')).collect())

# same work, healthy key: invoice_id has high cardinality -> spreads evenly
w_ok = Window.partitionBy('invoice_id').orderBy('sale_date')
timed('window over invoice_id (well distributed)',
      lambda: inv.withColumn('rn', F.row_number().over(w_ok))
                 .agg(F.max('rn')).collect())

In [0]:
%sql

select * from car_workshop.fact.fact_invoices limit 1000

In [0]:
%sql

select * from car_workshop.fact.fact_sales_transactions limit 1000

In [0]:
# skew a REAL FK: 80% of invoices point to one hot transaction
trx = spark.table(f'{CATALOG}.fact.fact_sales_transactions')

# invoices have NO transaction_id - the FK is polymorphic (source_type + source_id),
# so keep only the 'sales' branch and give the key its real name
inv_base = (spark.table(f'{CATALOG}.fact.fact_invoices')
            .filter("source_type = 'sales'")                      # ~85% of invoices
            .withColumnRenamed('source_id', 'transaction_id'))

# any existing id will do as the hot key - 80% of rows get overwritten anyway
hot_trx = inv_base.select('transaction_id').first()['transaction_id']

(inv_base
 .withColumn('transaction_id',
             F.when(F.rand(seed=42) < 0.8, F.lit(hot_trx)).otherwise(F.col('transaction_id')))
 .write.mode('overwrite').saveAsTable(f'{LAB}.invoices_hot_fk'))

# both sides are large facts -> no broadcast escape, real shuffle join
timed('fact-to-fact join on skewed FK',
      lambda: spark.table(f'{LAB}.invoices_hot_fk')
                   .join(trx, 'transaction_id')
                   .agg(F.sum('value_gross')).collect())

In [0]:
%sql

select * from car_workshop.lab.invoices_hot_fk limit 1000

In [0]:
inv = spark.table(f'{LAB}.fact_invoices')

# stage 1: pre-aggregate per (key, salt) -> stage 2: final agg per key
salted = (inv
    .withColumn('salt', (F.rand(seed=7) * 32).cast('int'))
    .groupBy('skew_test', 'salt').agg(F.sum('value_gross').alias('partial'))
    .groupBy('skew_test').agg(F.sum('partial').alias('total')))

# correctness check - salting must NOT change the numbers
plain = inv.groupBy('skew_test').agg(F.sum('value_gross').alias('total'))
display(salted.orderBy('skew_test'))
display(plain.orderBy('skew_test'))


In [0]:

# SERVERLESS variant - salting is plain DataFrame code, so exercise CORRECTNESS:
# the salted join must return exactly the same aggregate as the plain join
# (this is the part people get wrong - forgetting the dim replication).
# The performance effect needs a forced SMJ -> classic cluster.
SALT = 16 # why 16 ?
skewed_df = spark.table(f'{LAB}.skewed_sales_items')
products_df = spark.table(f'{CATALOG}.dim.dim_products')

fact_salted_sl = (skewed_df
    .withColumn('salt', (F.rand(seed=7) * SALT).cast('int'))
    .withColumn('k_salt', F.concat_ws('_', 'product_id', 'salt')))

dim_salted_sl = (products_df
    .crossJoin(spark.range(SALT).withColumnRenamed('id', 'salt'))
    .withColumn('k_salt', F.concat_ws('_', 'product_id', 'salt'))
    .drop('product_id', 'salt'))

plain = skewed_df.join(products_df, 'product_id').agg(F.round(F.sum('value_net'), 2)).first()[0]
salted = fact_salted_sl.join(dim_salted_sl, 'k_salt').agg(F.round(F.sum('value_net'), 2)).first()[0]
print(f'plain join:  {plain:,}')
print(f'salted join: {salted:,}  -> identical: {plain == salted}')

In [0]:
fact_salted_sl.display()

In [0]:
dim_salted_sl.display()

In [0]:
# two-stage (salted) AGGREGATION - for distributive aggs: sum/count/min/max
# stage 1: partial agg on (key, salt) -> stage 2: final agg on key
partial = (skewed
    .withColumn('salt', (F.rand(seed=7) * SALT).cast('int'))
    .groupBy('product_id', 'salt')
    .agg(F.sum('value_net').alias('partial_sum'), F.count('*').alias('partial_cnt')))

final = (partial.groupBy('product_id')
    .agg(F.sum('partial_sum').alias('revenue'),
         (F.sum('partial_sum') / F.sum('partial_cnt')).alias('avg_value')))  # avg = sum/count!

display(final.orderBy(F.desc('revenue')).limit(5))
# note: count(distinct) can NOT be salted this way

In [0]:
# SERVERLESS variant - hybrid salting without the conf toggles: verify the logic
# (row counts must match the plain join), benchmark later on the classic cluster.
SALT = 16
skewed_df = spark.table(f'{LAB}.skewed_sales_items')
products_df = spark.table(f'{CATALOG}.dim.dim_products')
salt_range_sl = spark.range(SALT).withColumnRenamed('id', 'salt')

counts_sl = skewed_df.groupBy('product_id').agg(F.count('*').alias('cnt'))
median_sl = counts_sl.select(F.expr('percentile_approx(cnt, 0.5)')).first()[0]
hot_sl = counts_sl.filter(F.col('cnt') > 20 * median_sl).select('product_id')

fact2_sl = (skewed_df
    .join(F.broadcast(hot_sl.withColumn('is_hot', F.lit(True))), 'product_id', 'left')
    .withColumn('salt', F.when(F.col('is_hot').isNotNull(),
                               (F.rand(seed=7) * SALT).cast('int')).otherwise(F.lit(0)))
    .withColumn('k_salt', F.concat_ws('_', 'product_id', 'salt')))

dim_hot_sl = products_df.join(F.broadcast(hot_sl), 'product_id', 'inner').crossJoin(salt_range_sl)
dim_cold_sl = products_df.join(F.broadcast(hot_sl), 'product_id', 'left_anti').withColumn('salt', F.lit(0))
dim2_sl = (dim_hot_sl.unionByName(dim_cold_sl)
    .withColumn('k_salt', F.concat_ws('_', 'product_id', 'salt'))
    .drop('product_id', 'salt'))

plain_cnt = skewed_df.join(products_df, 'product_id').count()
hybrid_cnt = fact2_sl.join(dim2_sl, 'k_salt').count()
print(f'plain: {plain_cnt:,} rows, hybrid-salted: {hybrid_cnt:,} -> identical: {plain_cnt == hybrid_cnt}')